# Tutorial Notebook: Working with the pelmesha Package
To install a Python package, the standard approach is to use `pip`

In [ ]:
!pip install pelmesha

## Constructing a DataSet Instance for Joint Data Processing

Studying biological samples in mass spectrometry, especially with the application of ML and AI, requires meticulous and fine-grained data processing. This need arises from the persistent contributions of systematic and random instrument drift, which are unique to each instrument and its components, as well as to the nature and conditions of its operation. These factors lead to instability of the instrument's calibration both within and across experiments. To build a satisfactory feature matrix for samples where the difference between molecules exceeds 1 m/z in modern instruments, it is sufficient to align the calibration and set a tolerance for random signal drift. In real biological sample data, however, the difference between molecules is at the very limit of both the instrument's resolving power and random signal drift.

This package introduces a novel algorithm for eliminating random drift. It is based on statistics, peak characteristics, and the discretization of the m/z scale in profile mass spectra to construct a peak density distribution via the KDE method. The algorithm performs careful, dynamic, and fully automated binning of the mass spectrum while discarding empty regions. At the same time, it is less sensitive to systematic drift and to the risk of merging signals from multiple molecules, so it can be used even without mass spectrum alignment — though preferably within a single experiment only or a series of experiments on the same instrument with minimal instability of the instrument's calibration.

## Dataset Configuration
First, we create an empty DataSet instance without passing any arguments during initialization. However, file paths can also be provided directly at this stage with configs.

In [1]:
from pelmesha.serving import DataSet

dataset = DataSet()

DataSet is initialized. Current data samples:
DataSet (empty)


Our dataset is currently empty. Now we’ll pass the path to a folder (or a single file) to the DataSet instance, then display its contents along with their configurations. We’ll also show the ROI configs for one of the sources — at the moment, they’re all identical for all samples and ROIs.

In [3]:
path = r"D:\Testing\Our_data\Rapiflex" #Use your own path

dataset.add_sources(path) # or dataset(path)
display(dataset)
display(dataset['roi1_e046'].roi_configs)

c:\Job_and_Literature\Programming\Python\projects\pelmesha\.venv\Lib\site-packages\pyimzml\ontology\ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size". Updating name to "pixel size (x)".
  warn(
c:\Job_and_Literature\Programming\Python\projects\pelmesha\.venv\Lib\site-packages\pyimzml\ontology\ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size". Updating name to "pixel size (x)".
  warn(
c:\Job_and_Literature\Programming\Python\projects\pelmesha\.venv\Lib\site-packages\pyimzml\ontology\ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size". Updating name to "pixel size (x)".
  warn(


Sample name,Mass spectra number,Continious,Peaklists,Processed mass spectra,Peaks density,Directory,Previous configs
roi1_e046,26370,Yes,Yes,No,Yes,D:\Testing\Our_data\Rapiflex\roi1_e046,Open
roi3_e047,17357,Yes,Yes,No,Yes,D:\Testing\Our_data\Rapiflex\roi3_e047,Open
roi8_e040,21158,Yes,Yes,No,Yes,D:\Testing\Our_data\Rapiflex\roi8_e040,Open


{'R00': Configs(
   functions:
     resample_mz_scale: resample_mz_step=None, resample_num_points=None
     modify_raw_spectrum: zero_points_to_peaks_ext=False, mz_segments_to_zero=None
     smoothing: smooth_algo=None, smooth_window=7, smooth_cycles=1
     msalign: align_peaks=None, align_method='cubic', align_width=10, align_ratio=2.5, align_resolution=100, align_iterations=5, align_grid_steps=20, align_shift_range=[-0.95, 0.95], align_pweights=None, return_shifts=False, align_by_index=False, only_shift=False
     peakpicker: fwhhfilter=None, oversegmentationfilter=None, heightfilter=None, rel_heightfilter=None, peaklocation=1, noise_func=np.std, noise_est_iterations=3, SNR_threshold=3, Calc_peak_area=True, headers=['spectra_ind', 'mz', 'Intensity', 'Area', 'SNR', 'PextL', 'PextR', 'FWHML', 'FWHMR', 'Noise', 'Mean noise']
   methods:
     Baseline:
       __init__(check_finite=True, output_dtype=None)
       asls(lam=1000000.0, p=0.01, diff_order=2, max_iter=50, tol=0.001, weights=No

We can see one of the added sources and its processing settings — by default, everything is set to the default values. Each source is wrapped in a `PreparedDataSource` and registered by its sample name

Per-source processing and KDE configurations can be adjusted either for all ROIs at once or for a single ROI:

In [ ]:
# Update a parameter for all ROIs of one sample
dataset["roi1_e046"].update({"smooth_window": 7})
dataset["roi1_e046"].update_kde(bwc=1.1)

# Or configure a specific ROI directly
configs_roi_R00 = dataset['roi1_e046'].roi_configs['R00']
configs_roi_R00['SNR_threshold'] = 4
configs_roi_R00['smooth_algo'] = 'GA'
configs_roi_R00['smooth_window'] = 5
configs_roi_R00['lam'] = 500000

kde_config = dataset["roi1_e046"].roi_kde_configs["R00"]
kde_config["bwc"] = 1.1

display(dataset['roi1_e046'].roi_configs)

An alternative way to configure settings is by using the `PipelineConfigurator` class. This class also enables users to incorporate custom mass spectrum processing functions. However, this approach is currently intended for advanced users only, as it has certain limitations and requires further development.

The resulting configuration can be applied to multiple samples or data sources, which supports consistent processing across different datasets.

In [ ]:
from pelmesha.cookbook import PipelineConfigurator
configs = PipelineConfigurator()
configs['align_peaks'] = [128,768,769]
configs['smooth_algo'] = 'GA'
configs['smooth_window'] = 5
configs['SNR_threshold'] = 4
configs['resample_num_points'] = 75000
configs['align_shift_range'] = [-0.5, 0.5]
configs.set_method('Baseline','modpoly') #замена метода коррекции базовой линии на другой - у него также меняются параметры
# Если необходимо отключить обработку базовой линии, то его следует удалить из конфигов
configs.delete('Baseline')
display(configs)

Now we can either pass the configuration directly to multiple samples, or create a new dataset and set it as the default configuration for all samples at once.

In [ ]:

from pelmesha.serving import DataSet
# Вариант 1: обновить конфиги части данных
dataset['roi1_e046'].roi_configs['R00'] = configs
dataset['roi8_e040'].roi_configs['R00'] = configs
display(dataset['roi1_e046'].roi_configs)

# Вариант 2: создать новый датасет и сразу ему передать конфиги
path = r"D:\Testing\Our_data\Rapiflex"

dataset_2 = DataSet()
dataset_2.add_sources(path, configs)
display(dataset_2['roi3_e047'].roi_configs)


## Running the Processing Pipeline
Before executing the main processing run, it’s advisable to check what the output will roughly look like with the current settings and whether it meets our expectations. We’ll use the `audit_processing` method for this purpose.

In [ ]:
dataset.audit_processing("roi8_e040",draw_mz_range=(880,920), draw_spectrum_idx=5000)

If the audit result isn’t satisfactory, it’s better to fine‑tune the settings accordingly.

Once the desired outcome is achieved, the dataset provides two methods for running the data processing: `process` and `peakpick`.

The `process` method produces HDF5 files containing the mass spectra — note that the spectra must be continuous (or resampling should be configured). These files are stored next to each source as `*_processed_spectra.hdf5`. They contain matrices of shape (N, M), where N is the number of spectra and M is the number of mass spectrum points.

<div class="alert alert-block alert-info">
<b>Note:</b> The process method is largely provided as a baseline operation and may be useful for users who need an alternative data‑handling approach — for example, for custom binning workflows or other downstream processing strategies. However, <b>this method is not required for the subsequent creation of the feature matrix via the KDE method</b>.
</div>

The `peakpick` method writes peak lists to an HDF5 files next to each source as `*_peaklists.hdf5`. The data is stored as a matrix of shape (N, M). Here, M represents the peak properties (including the corresponding mass spectrum index), and N is the total number of peaks across all mass spectra.

In [ ]:
from pelmesha.serving import DataSet
from pelmesha.cookbook import PipelineConfigurator

configs = PipelineConfigurator()
configs['smooth_algo'] = 'GA'
configs['smooth_window'] = 5
configs['SNR_threshold'] = 5
path = r"D:\Testing\Our_data\Rapiflex"
dataset = DataSet()
dataset.add_sources(path, configs)
dataset.peakpick()

Once we have the peak lists, we can use this data to construct the peak density distribution for each ROI. This step is key for feature. For example, to generate KDE‑based feature matrices that capture the local intensity and distribution of peaks within each m/z region.

In [ ]:
from pelmesha import DataSet
from pelmesha.cookbook import KDEConfigs
path = r"D:\Testing\Our_data\Rapiflex"
kde_configs = KDEConfigs(KDE_algo="tree",bwc=1)
data = DataSet(path,kde_configs=kde_configs)
data.estimate_peak_density_kde()

Now that we have the individual peak PDFs and the peak lists, we can construct feature matrices in any combination.  
##### Feature matrix options
When building the feature matrix, you can configure additional behaviors:
- `pivot_values` - pivoting on the required peak characteristics (e.g., area, intensity, SNR).
- `countf` or `rel_countf` - peak filtering by their occurrence across the whole dataset (absolute or relative count).
- `duplication_drop` - creation of consensus peaks.
- `local_roi_idx`- Index display: configure whether the feature matrix index shows the absolute index in the file or the relative index within the ROI.
- `merge_with_coords` - Merging coordinates/RT into the features.
- `save_path` - Optional saving of the resulting data to Parquet format.

In [ ]:
from pelmesha import DataSet
path = r"D:\Testing\Our_data\Rapiflex"

data = DataSet(path)
data.feature_matrix(save_path=r"D:\Testing\Our_data\Rapiflex\feature_matrix",pivot_values=['Area'], countf=25, draw_borders = 3, duplicates_drop=True, local_roi_idx=False, merge_with_coords=True)

The key benefit of code design: once the peak PDFs and peak lists are computed, any combination of sources can be assembled on‑the‑fly, and re‑running the expensive processing step (e.g., peak picking or KDE computation) is not required. This makes experimentation with different source groupings fast and efficient.

Since a data source may contain multiple ROIs, the feature_matrix method also provides arguments for selecting the required ROIs: `samples`, `rois`, and `sample_rois_map`.

- `samples` — the list of data sources.
- `rois` — when the data is homogeneous across ROIs, you can use only the specified ROIs for all samples.
- `sample_rois_map` — a detailed map specifying which data source to use and which ROIs to extract from it.

This granular control is especially useful when dealing with data acquired in different modes (positive/negative) or with heterogeneous ROI layouts across samples.

Below is an example of a complex configuration for a complex dataset.

In [ ]:
from pelmesha import DataSet

# Example of a complex configuration where each ROI is treated as a separate channel
# ROIs 0, 1 → negative and positive modes in ion trap mass analyzer
# ROIs 2, 3 → negative and positive modes in Orbitrap mass analyzer

# Parameter for zeroing induced spurious signal in mass spectra
signal_distortions = [(99, 100.2),
                      (102.09, 103.86),
                      (107.79, 109.4),
                      (112.65, 112.9),
                      (123.2, 123.6),
                      (127, 128.2),
                      (135.4, 136.12), 
                     (155.7, 156.6), 
                     (168.5, 170.8),
                     (188.75, 189.15),
                     (202.00, 202.55),
                     (203.13, 204.65),
                     (206.4, 206.8), 
                     (210.30, 210.60), 
                     (220.10, 221.02), 
                     (229.36, 231.53), 
                     (241.92, 243.0),
                     (277.4, 278.4), 
                     (285.83, 287.11),
                     (321.6, 322.0),
                     (375.0, 375.6),
                     (398.18, 401.15), 
                     (472.29, 472.89), 
                     (508.34, 512.61), 
                     (620.00, 622.81), 
                     (643.60, 645.03), 
                     (674.0, 676.17), 
                     (677.37, 683.2), 
                     (906.92, 908.83), 
                     (510.0, 511.2), 
                     (622.5, 627.5), 
                     (643.0, 644.5), 
                     (676.0, 679.5)]


path = r"C:\Job_and_Literature\Esi_test"
# Create a dataset with common parameters for all sources, which we’ll later fine‑tune for individual ROIs.
data = DataSet(path, KDE_algo="tree",bwc=1,SNR_threshold=5,smooth_algo='GA', align_shift_range = (-0.25,0.25), zero_points_to_peaks_ext=True, resample_mz_step = 0.0125)
data.set_reference_source(data.sources['Esi_test_3_t'], KDE_algo = 'tree', bwc = 1, SNR_threshold = 10, smooth_algo = 'GA', zero_points_to_peaks_ext=True)
for source in data.sources.values():
    for roi in source.roi_configs.keys():
        if roi in ["2","3"]: #Fine-tuning for ROIs 2 and 3
            source.roi_configs[roi].delete('Baseline')
            source.roi_configs[roi]['resample_mz_step'] = 0.006125
            source.roi_configs[roi]['align_shift_range'] = (-0.15,0.15)
            source.roi_configs[roi]['mz_segments_to_zero'] = signal_distortions
ref_source = data.reference_source
for roi in ref_source.roi_configs.keys():
    if roi in ["2","3"]: #Fine-tuning for ROIs 2 and 3 of the reference source
        ref_source.roi_configs[roi].delete('Baseline')
        ref_source.roi_configs[roi]['resample_mz_step'] = 0.006125
        ref_source.roi_configs[roi]['align_shift_range'] = (-0.15,0.15)
        ref_source.roi_configs[roi]['mz_segments_to_zero'] = signal_distortions


rois = ['0','1',"2","3"]
for roi in rois: # Setting the reference peaks for each ROI
    data.get_reference_peaks(roi = roi,step=500,num_peaks_per_step=3)
    data.set_align_peaks_from_ref(rois = roi)
data.peakpick(draw_mz_range=(750,800))
data.estimate_peak_density_kde(draw_borders= 3)

# Creating feature matrix for each ROI/channel
rois = ['0','1',"2","3"]
features = {}
for roi in rois:
    features[roi] = data.feature_matrix(rois = roi, countf=25, draw_borders= 3)